# Bu Dersi Google Colab'da Çalıştır

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BILSEM-BT/Python/blob/main/25-PythonKMeansKumeleme.ipynb)

Bu notebook GitHub üzerinde ders dokümanı olarak yayımlanır. Kodları çalıştırmak ve üzerinde denemeler yapmak için yukarıdaki **Open in Colab** butonunu kullanabilirsiniz.

### Nasıl çalışacağız?

1. **Open in Colab** butonuna tıklayın.
2. Açılan notebook'taki kod hücrelerini `▶` düğmesiyle çalıştırın.
3. Kodları değiştirerek farklı sonuçları deneyin.
4. Çalışmalarınız kendi Colab çalışma alanınızda tutulur; bu GitHub'daki ana ders dosyasını değiştirmez.

> **Önemli:** GitHub'daki bu dosya dersin ana ve değiştirilmeyen kaynağıdır. Colab'da yaptığınız değişiklikler bu dosyaya otomatik olarak yazılmaz.

---

# 25 - Python'da Denetimsiz Öğrenme ve K-Means Kümeleme

**Niyazi Sayın BİLSEM**  
**Bilişim Teknolojileri Dersi**  
**Ders Öğretmeni: Ersin ŞANLI**

Önceki yapay zeka derslerinde elimizde bir hedef değişken bulunuyordu.

Örneğin:

- sınıf etiketi,
- sınav puanı,
- fiyat,
- durum bilgisi.

Bu derste farklı bir problem türüne geçiyoruz.

Artık modelimize doğru cevapları vermeyeceğiz.

Model, verilerin içindeki benzerlikleri kendisi keşfetmeye çalışacak.

Bu yaklaşıma **denetimsiz öğrenme** denir.

Bu dersin temel algoritması **K-Means Kümeleme** olacaktır.

Ders sonunda öğrencinin:

- denetimli ve denetimsiz öğrenme farkını açıklayabilmesi,
- kümeleme problemini tanıyabilmesi,
- K-Means mantığını anlayabilmesi,
- özellikleri ölçeklendirebilmesi,
- `fit()` ve `fit_predict()` kullanabilmesi,
- cluster etiketlerini inceleyebilmesi,
- cluster merkezlerini yorumlayabilmesi,
- inertia kavramını anlayabilmesi,
- Elbow yöntemi uygulayabilmesi,
- silhouette score hesaplayabilmesi,
- uygun K değerini karşılaştırabilmesi,
- yeni bir örneği kümeye atayabilmesi,
- cluster profilleri oluşturabilmesi,
- kümeleme sonucunu dosyaya kaydedebilmesi

hedeflenmektedir.


# 1. Denetimli Öğrenmeyi Hatırlayalım

Önceki derslerde:

```text
X → Özellikler
y → Doğru cevap / hedef
```

vardı.

Örneğin:

```text
Sensör Değerleri
↓
Normal / İncelenmeli
```

Burada doğru sınıflar eğitim verisinde zaten bulunuyordu.

Bu nedenle model **denetimli** olarak öğreniyordu.


# 2. Denetimsiz Öğrenme Nedir?

Denetimsiz öğrenmede hedef değişken yoktur.

Yalnızca:

```text
X → Özellikler
```

bulunur.

Model verilerdeki:

- benzerlikleri,
- farklılıkları,
- grupları,
- örüntüleri

keşfetmeye çalışır.


# 3. Kümeleme Nedir?

Benzer özelliklere sahip örnekleri aynı grupta toplamaya **kümeleme** denir.

Örnek kullanım alanları:

- müşteri segmentasyonu,
- sensör davranışlarının gruplanması,
- ürün grupları,
- belge gruplama,
- görüntü renklerinin gruplanması,
- etkinlik katılım örüntüleri,
- kullanıcı davranış analizi.

Kümeleme sonucundaki gruplara **cluster / küme** denir.


# 4. Önemli Uyarı: Küme Etiketi Gerçek Sınıf Değildir

K-Means bize:

```text
0
1
2
```

gibi küme numaraları verebilir.

Bu numaralar:

- başarı seviyesi,
- yetenek seviyesi,
- kalite notu,
- resmi kategori

anlamına gelmez.

Küme numaraları algoritmanın oluşturduğu grupların teknik etiketleridir.

Özellikle öğrenci verileri üzerinde kümeleme yaparken sonuçları öğrencileri etiketlemek veya yeteneklerine ilişkin kesin yargı üretmek için kullanmamalıyız.


# 5. Ders Senaryosu

Örnek bir BİLSEM atölyesinde anonim ve tamamen yapay çalışma verilerimiz olduğunu düşünelim.

Her kayıt için:

- haftalık kodlama süresi,
- tamamlanan proje sayısı,
- devam oranı,
- uygulama puanı

bulunuyor.

Amacımız:

```text
Benzer çalışma örüntülerine sahip kayıtlar var mı?
```

sorusunu incelemek.

Bu veri gerçek öğrencilere ait değildir; yalnızca eğitim amacıyla üretilmiştir.


# 6. Gerekli Kütüphaneler

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# 7. Yapay Veri Kümesini Oluşturmak

Veri kümesini üç farklı davranış örüntüsü etrafında üreteceğiz.

Ancak analiz sırasında bu gizli grupları modele vermeyeceğiz.


In [ ]:
rng = np.random.default_rng(42)

ornek_sayisi = 60

def grup_uret(ortalamalar, sapmalar):
    return np.column_stack([
        rng.normal(
            ortalamalar[i],
            sapmalar[i],
            ornek_sayisi
        )
        for i in range(4)
    ])

grup_1 = grup_uret(
    [2, 1, 65, 55],
    [0.6, 0.4, 3, 4]
)

grup_2 = grup_uret(
    [6, 3, 82, 75],
    [0.8, 0.5, 3, 4]
)

grup_3 = grup_uret(
    [11, 6, 97, 95],
    [0.8, 0.5, 2, 3]
)

tum_veri = np.vstack([
    grup_1,
    grup_2,
    grup_3
])

df = pd.DataFrame(
    tum_veri,
    columns=[
        "HaftalikKodlamaSaati",
        "ProjeSayisi",
        "DevamOrani",
        "UygulamaPuani"
    ]
)

df["HaftalikKodlamaSaati"] = (
    df["HaftalikKodlamaSaati"]
    .clip(0.5, 15)
    .round(1)
)

df["ProjeSayisi"] = (
    df["ProjeSayisi"]
    .clip(0, 8)
    .round()
    .astype(int)
)

df["DevamOrani"] = (
    df["DevamOrani"]
    .clip(50, 100)
    .round(1)
)

df["UygulamaPuani"] = (
    df["UygulamaPuani"]
    .clip(30, 100)
    .round(1)
)

df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

df.head()


# 8. Veri Kümesinin Boyutu

In [ ]:
print(
    "Satır ve sütun:",
    df.shape
)


# 9. İlk 10 Kayıt

In [ ]:
df.head(10)


# 10. Veri Türleri

In [ ]:
print(df.dtypes)


# 11. Eksik Veri Kontrolü

In [ ]:
print(
    df.isna().sum()
)


# 12. Tekrar Eden Kayıt Kontrolü

In [ ]:
print(
    "Tekrar eden satır:",
    df.duplicated().sum()
)


# 13. Temel İstatistikler

In [ ]:
df.describe()


# 14. Özellik Dağılımlarını İncelemek

İlk özellik:


In [ ]:
plt.hist(
    df["HaftalikKodlamaSaati"],
    bins=15
)

plt.xlabel("Haftalık Kodlama Saati")
plt.ylabel("Kayıt Sayısı")
plt.title("Kodlama Süresi Dağılımı")
plt.show()


# 15. Uygulama Puanı Dağılımı

In [ ]:
plt.hist(
    df["UygulamaPuani"],
    bins=15
)

plt.xlabel("Uygulama Puanı")
plt.ylabel("Kayıt Sayısı")
plt.title("Uygulama Puanı Dağılımı")
plt.show()


# 16. İki Özelliğin İlişkisini Görmek

In [ ]:
plt.scatter(
    df["HaftalikKodlamaSaati"],
    df["UygulamaPuani"]
)

plt.xlabel("Haftalık Kodlama Saati")
plt.ylabel("Uygulama Puanı")
plt.title("Kodlama Süresi ve Uygulama Puanı")
plt.show()


Grafikte bazı doğal gruplar olabileceğini gözlemleyebiliriz.

Ancak K-Means dört özelliği birlikte kullanacaktır.


# 17. K-Means Nasıl Çalışır?

Basitleştirilmiş K-Means süreci:

```text
1. K küme merkezi seç
2. Her örneği en yakın merkeze ata
3. Her kümenin yeni merkezini hesapla
4. Örnekleri yeniden merkezlere ata
5. Merkezler yeterince değişmeyene kadar tekrarla
```

K-Means uzaklık temelli bir algoritmadır.


# 18. K Değeri Nedir?

K-Means kullanırken kaç küme oluşturulacağını önceden belirtiriz.

Örneğin:

```python
KMeans(n_clusters=3)
```

üç küme oluşturmayı hedefler.

Ancak gerçek bir problemde doğru K değerini her zaman önceden bilmiyor olabiliriz.

Bu nedenle ileride:

- Elbow yöntemi,
- silhouette score

kullanacağız.


# 19. Neden Ölçeklendirme Yapıyoruz?

Özelliklerimiz farklı ölçeklerde:

```text
ProjeSayisi        → yaklaşık 0-8
DevamOrani         → yaklaşık 50-100
UygulamaPuani      → yaklaşık 30-100
KodlamaSaati       → yaklaşık 0-15
```

K-Means uzaklık kullandığı için büyük ölçekli özellikler hesaplamayı daha fazla etkileyebilir.

Bu nedenle özellikleri standardize edeceğiz.


# 20. StandardScaler

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    df
)

print(
    X_scaled[:5]
)


StandardScaler her özellik için yaklaşık olarak:

```text
ortalama → 0
standart sapma → 1
```

ölçeğine dönüştürür.


# 21. Ölçeklendirilmiş Veriyi DataFrame Yapmak

In [ ]:
scaled_df = pd.DataFrame(
    X_scaled,
    columns=df.columns
)

scaled_df.head()


# 22. Ölçeklendirme Sonrası Ortalamalar

In [ ]:
print(
    scaled_df.mean()
)


# 23. Ölçeklendirme Sonrası Standart Sapmalar

In [ ]:
print(
    scaled_df.std(
        ddof=0
    )
)


# 24. İlk K-Means Modeli

İlk olarak K=3 deneyelim.


In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

kmeans.fit(
    X_scaled
)

print(
    "Model eğitildi."
)


Denetimsiz öğrenmede `fit()` sırasında hedef değişken vermedik.

Çünkü elimizde:

```text
y
```

yoktur.


# 25. Cluster Etiketleri

In [ ]:
print(
    kmeans.labels_[:20]
)


Her kayıt artık:

```text
0
1
2
```

kümelerinden birine atanmıştır.


# 26. Etiketleri Veri Kümesine Eklemek

In [ ]:
sonuc_df = df.copy()

sonuc_df["Kume"] = (
    kmeans.labels_
)

sonuc_df.head()


# 27. Her Kümede Kaç Kayıt Var?

In [ ]:
print(
    sonuc_df["Kume"]
    .value_counts()
    .sort_index()
)


# 28. Kümeleri İki Özellikle Görselleştirmek

In [ ]:
for kume in sorted(
    sonuc_df["Kume"].unique()
):
    secim = (
        sonuc_df["Kume"]
        == kume
    )

    plt.scatter(
        sonuc_df.loc[
            secim,
            "HaftalikKodlamaSaati"
        ],
        sonuc_df.loc[
            secim,
            "UygulamaPuani"
        ],
        label=f"Küme {kume}"
    )

plt.xlabel("Haftalık Kodlama Saati")
plt.ylabel("Uygulama Puanı")
plt.title("K-Means Kümeleri")
plt.legend()
plt.show()


Bu grafik yalnızca iki özelliği göstermektedir.

K-Means kümeleri dört özellik birlikte kullanılarak oluşturulmuştur.


# 29. Cluster Merkezleri

K-Means her küme için bir merkez noktası öğrenir.


In [ ]:
print(
    kmeans.cluster_centers_
)


Ancak bu merkezler ölçeklendirilmiş uzaydadır.

İnsan tarafından daha kolay yorumlamak için orijinal ölçeğe geri dönüştürebiliriz.


# 30. Küme Merkezlerini Orijinal Ölçeğe Döndürmek

In [ ]:
merkezler_orijinal = (
    scaler.inverse_transform(
        kmeans.cluster_centers_
    )
)

merkez_df = pd.DataFrame(
    merkezler_orijinal,
    columns=df.columns
)

merkez_df.index.name = "Kume"

merkez_df.round(2)


# 31. Küme Profilleri

Merkezler bize kümelerin genel profilini yorumlama fırsatı verir.

Örneğin bir kümede:

- kodlama saati daha yüksek,
- proje sayısı daha yüksek,
- devam oranı daha yüksek

olabilir.

Ancak bu yorumlar yalnızca veri içindeki örüntüyü açıklar.

İnsanları kalıcı veya değer yüklü kategorilere ayırmak için kullanılmamalıdır.


# 32. Gerçek Küme Ortalamaları

K-Means merkezlerinin yanında kümelerdeki gerçek kayıtların ortalamalarını da hesaplayalım.


In [ ]:
kume_ortalamalari = (
    sonuc_df
    .groupby("Kume")[
        [
            "HaftalikKodlamaSaati",
            "ProjeSayisi",
            "DevamOrani",
            "UygulamaPuani"
        ]
    ]
    .mean()
)

kume_ortalamalari.round(2)


# 33. Küme Profillerini Grafikleştirmek

Örnek olarak haftalık kodlama saatlerini karşılaştıralım.


In [ ]:
plt.bar(
    kume_ortalamalari.index.astype(str),
    kume_ortalamalari[
        "HaftalikKodlamaSaati"
    ]
)

plt.xlabel("Küme")
plt.ylabel("Ortalama Kodlama Saati")
plt.title("Kümelere Göre Kodlama Süresi")
plt.show()


# 34. Küme Uygulama Puanları

In [ ]:
plt.bar(
    kume_ortalamalari.index.astype(str),
    kume_ortalamalari[
        "UygulamaPuani"
    ]
)

plt.xlabel("Küme")
plt.ylabel("Ortalama Uygulama Puanı")
plt.title("Kümelere Göre Uygulama Puanı")
plt.show()


# 35. Cluster Numaraları Sıralı Değildir

Küme:

```text
0
1
2
```

etiketleri arasında doğal bir büyüklük ilişkisi yoktur.

Örneğin:

```text
Küme 2 > Küme 1
```

gibi bir anlam çıkarılamaz.

Modeli farklı `random_state` veya veriyle çalıştırdığınızda benzer grupların numaraları değişebilir.


# 36. `fit_predict()`

Modeli eğitip cluster etiketlerini tek adımda almak mümkündür.


In [ ]:
kmeans_ornek = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

etiketler = (
    kmeans_ornek.fit_predict(
        X_scaled
    )
)

print(
    etiketler[:10]
)


# 37. Inertia Nedir?

K-Means'in önemli ölçülerinden biri `inertia_` değeridir.

Basit olarak her noktanın kendi küme merkezine olan karesel uzaklıklarının toplamını temsil eder.

Düşük inertia, noktaların merkezlere daha yakın olduğunu gösterir.


In [ ]:
print(
    "Inertia:",
    kmeans.inertia_
)


# 38. Inertia Tek Başına Kullanılır mı?

K arttıkça kümeler küçülür ve inertia doğal olarak düşme eğilimindedir.

Örneğin:

```text
K = 1 → yüksek inertia
K = 2 → daha düşük
K = 3 → daha düşük
...
```

Bu nedenle yalnızca en düşük inertia değerine bakarak K seçemeyiz.

Aksi halde K'yı gereksiz yere çok büyütürüz.


# 39. Elbow Yöntemi

Farklı K değerleri için inertia hesaplarız.

Grafikte iyileşmenin belirgin biçimde yavaşlamaya başladığı bölgeye **dirsek / elbow** denir.

Bu değer uygun K için aday olabilir.


# 40. K=1 ile K=9 Arasında Inertia Hesaplamak

In [ ]:
inertia_sonuclari = []

for k in range(1, 10):
    model_k = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model_k.fit(
        X_scaled
    )

    inertia_sonuclari.append({
        "K": k,
        "Inertia": model_k.inertia_
    })

inertia_df = pd.DataFrame(
    inertia_sonuclari
)

inertia_df


# 41. Elbow Grafiği

In [ ]:
plt.plot(
    inertia_df["K"],
    inertia_df["Inertia"],
    marker="o"
)

plt.xlabel("K")
plt.ylabel("Inertia")
plt.title("Elbow Yöntemi")
plt.grid()
plt.show()


Bu veri kümesinde grafikte yaklaşık üç küme civarında belirgin bir kırılma görmeyi bekleriz.

Elbow yöntemi her veri kümesinde kesin ve net bir dirsek üretmek zorunda değildir.


# 42. Silhouette Score Nedir?

Silhouette score bir örneğin:

- kendi kümesine ne kadar uyumlu,
- diğer kümelerden ne kadar ayrılmış

olduğunu değerlendirmeye yardımcı olur.

Genel aralık:

```text
-1 ile +1
```

arasındadır.

Daha yüksek değerler genel olarak daha iyi ayrılmış kümelere işaret eder.


# 43. Silhouette Score Kullanım Şartı

Silhouette score için en az iki farklı cluster etiketi gerekir.

Bu nedenle:

```text
K = 1
```

için silhouette score hesaplanmaz.


# 44. K=2 ile K=8 Arasında Silhouette Score

In [ ]:
silhouette_sonuclari = []

for k in range(2, 9):
    model_k = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    etiket = model_k.fit_predict(
        X_scaled
    )

    skor = silhouette_score(
        X_scaled,
        etiket
    )

    silhouette_sonuclari.append({
        "K": k,
        "Silhouette": skor
    })

silhouette_df = pd.DataFrame(
    silhouette_sonuclari
)

silhouette_df


# 45. Silhouette Score Grafiği

In [ ]:
plt.plot(
    silhouette_df["K"],
    silhouette_df["Silhouette"],
    marker="o"
)

plt.xlabel("K")
plt.ylabel("Silhouette Score")
plt.title("K Değerine Göre Silhouette Score")
plt.grid()
plt.show()


# 46. En Yüksek Silhouette Değerini Bulmak

In [ ]:
en_iyi_satir = (
    silhouette_df.loc[
        silhouette_df[
            "Silhouette"
        ].idxmax()
    ]
)

print(en_iyi_satir)


Bu veri kümesi özel olarak üç belirgin grup içerecek şekilde oluşturulduğu için silhouette analizinde K=3 güçlü bir aday olmalıdır.


# 47. Elbow ve Silhouette Birlikte Kullanmak

K seçerken tek bir yönteme bağlı kalmak yerine:

- elbow grafiği,
- silhouette score,
- cluster büyüklükleri,
- cluster profillerinin anlamlılığı,
- problem bilgisi

birlikte değerlendirilebilir.

Kümeleme sonuçları problem bağlamından bağımsız yorumlanmamalıdır.


# 48. K=2 Sonucunu İncelemek

In [ ]:
model_k2 = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

etiket_k2 = model_k2.fit_predict(
    X_scaled
)

print(
    "K=2 Silhouette:",
    silhouette_score(
        X_scaled,
        etiket_k2
    )
)


# 49. K=3 Sonucunu İncelemek

In [ ]:
model_k3 = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

etiket_k3 = model_k3.fit_predict(
    X_scaled
)

print(
    "K=3 Silhouette:",
    silhouette_score(
        X_scaled,
        etiket_k3
    )
)


# 50. K=4 Sonucunu İncelemek

In [ ]:
model_k4 = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

etiket_k4 = model_k4.fit_predict(
    X_scaled
)

print(
    "K=4 Silhouette:",
    silhouette_score(
        X_scaled,
        etiket_k4
    )
)


# 51. Üç K Değerini Karşılaştırmak

In [ ]:
k_karsilastirma = pd.DataFrame({
    "K": [2, 3, 4],
    "Silhouette": [
        silhouette_score(
            X_scaled,
            etiket_k2
        ),
        silhouette_score(
            X_scaled,
            etiket_k3
        ),
        silhouette_score(
            X_scaled,
            etiket_k4
        )
    ]
})

k_karsilastirma


# 52. En İyi K ile Nihai Kümeleme

Bu örnekte K=3 kullanarak nihai cluster etiketlerini oluşturalım.


In [ ]:
final_kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

final_cluster = (
    final_kmeans.fit_predict(
        X_scaled
    )
)

analiz_df = df.copy()

analiz_df["Kume"] = (
    final_cluster
)

analiz_df.head()


# 53. Kümelerin Büyüklükleri

In [ ]:
kume_buyuklukleri = (
    analiz_df["Kume"]
    .value_counts()
    .sort_index()
)

print(
    kume_buyuklukleri
)


# 54. Küme Büyüklüklerini Grafikleştirmek

In [ ]:
plt.bar(
    kume_buyuklukleri.index.astype(str),
    kume_buyuklukleri.values
)

plt.xlabel("Küme")
plt.ylabel("Kayıt Sayısı")
plt.title("Küme Büyüklükleri")
plt.show()


# 55. Nihai Küme Profilleri

In [ ]:
profil_df = (
    analiz_df
    .groupby("Kume")[
        [
            "HaftalikKodlamaSaati",
            "ProjeSayisi",
            "DevamOrani",
            "UygulamaPuani"
        ]
    ]
    .mean()
    .round(2)
)

profil_df


# 56. Küme Profillerini Normalize Edilmiş Olarak Görmek

Özelliklerin aynı ölçekte olmadığı durumlarda profil karşılaştırmasını standardize değerlerle de yapmak yararlı olabilir.


In [ ]:
profil_scaled = pd.DataFrame(
    X_scaled,
    columns=df.columns
)

profil_scaled["Kume"] = (
    final_cluster
)

profil_scaled.groupby(
    "Kume"
).mean().round(2)


# 57. Küme Merkezleri ile Profil Ortalamaları

K-Means merkezleri cluster profilinin matematiksel merkezidir.

StandardScaler'dan geri çevrilmiş merkezleri görelim.


In [ ]:
final_merkezler = (
    scaler.inverse_transform(
        final_kmeans.cluster_centers_
    )
)

final_merkez_df = pd.DataFrame(
    final_merkezler,
    columns=df.columns
)

final_merkez_df.index.name = "Kume"

final_merkez_df.round(2)


# 58. Kümeleri İsimlendirmeli miyiz?

Teknik olarak kümeler:

```text
0
1
2
```

şeklinde kalabilir.

Analiz raporunda gerekiyorsa nötr ve betimleyici isimler kullanılabilir.

Örneğin:

```text
Düşük Kodlama Süresi Örüntüsü
Orta Kodlama Süresi Örüntüsü
Yüksek Kodlama Süresi Örüntüsü
```

Ancak insanları:

```text
başarısız
zayıf
iyi öğrenci
kötü öğrenci
```

gibi değer yüklü etiketlerle sınıflandırmak doğru değildir.


# 59. Nötr Profil İsimleri Oluşturmak

Kümelerin ortalama kodlama süresine göre yalnızca bu veri setine özel betimleyici isimler oluşturalım.


In [ ]:
sirali_kumeler = (
    profil_df[
        "HaftalikKodlamaSaati"
    ]
    .sort_values()
    .index
    .tolist()
)

profil_isimleri = {
    sirali_kumeler[0]:
        "Daha Az Kodlama Süresi",
    sirali_kumeler[1]:
        "Orta Kodlama Süresi",
    sirali_kumeler[2]:
        "Daha Fazla Kodlama Süresi"
}

analiz_df["Profil"] = (
    analiz_df["Kume"]
    .map(profil_isimleri)
)

analiz_df.head()


Bu isimler yalnızca haftalık kodlama süresinin göreli düzeyini betimler.

Öğrencinin genel yeteneği, kapasitesi veya potansiyeli hakkında hüküm vermez.


# 60. Profil Dağılımı

In [ ]:
print(
    analiz_df["Profil"]
    .value_counts()
)


# 61. Yeni Bir Örneği Hangi Kümeye Atarız?

K-Means eğitimden sonra yeni örnekler için `predict()` kullanabilir.

Ancak yeni veri, modelin eğitiminde kullanılan **aynı scaler** ile dönüştürülmelidir.


In [ ]:
yeni_kayit = pd.DataFrame({
    "HaftalikKodlamaSaati": [7.0],
    "ProjeSayisi": [3],
    "DevamOrani": [84.0],
    "UygulamaPuani": [79.0]
})

yeni_scaled = scaler.transform(
    yeni_kayit
)

yeni_kume = final_kmeans.predict(
    yeni_scaled
)[0]

print(
    "Küme:",
    yeni_kume
)

print(
    "Profil:",
    profil_isimleri[
        yeni_kume
    ]
)


# 62. Yeni Kayıt Tahmin Fonksiyonu

In [ ]:
def profil_tahmin(
    kodlama_saati,
    proje_sayisi,
    devam_orani,
    uygulama_puani
):
    yeni = pd.DataFrame({
        "HaftalikKodlamaSaati": [
            kodlama_saati
        ],
        "ProjeSayisi": [
            proje_sayisi
        ],
        "DevamOrani": [
            devam_orani
        ],
        "UygulamaPuani": [
            uygulama_puani
        ]
    })

    yeni_scaled = scaler.transform(
        yeni
    )

    kume = final_kmeans.predict(
        yeni_scaled
    )[0]

    return {
        "kume": int(kume),
        "profil": profil_isimleri[
            kume
        ]
    }


In [ ]:
print(
    profil_tahmin(
        10,
        5,
        96,
        93
    )
)


# 63. K-Means Pipeline

Ölçeklendirme ve K-Means'i Pipeline içinde birleştirmek mümkündür.

Bu yapı yeni verinin doğru ön işlemden geçmesini kolaylaştırır.


In [ ]:
from sklearn.pipeline import Pipeline

kumeleme_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "kmeans",
        KMeans(
            n_clusters=3,
            random_state=42,
            n_init=10
        )
    )
])

kumeleme_pipeline.fit(
    df
)

print(
    kumeleme_pipeline
)


# 64. Pipeline ile Yeni Veri Tahmini

In [ ]:
pipeline_kume = (
    kumeleme_pipeline.predict(
        yeni_kayit
    )[0]
)

print(
    "Pipeline kümesi:",
    pipeline_kume
)


Pipeline kullanımı özellikle modeli uygulamaya taşırken scaler ile K-Means'in birlikte kalmasını sağlar.


# 65. PCA ile İki Boyuta İndirme

Verimiz dört özellikli olduğu için bütün yapıyı iki boyutlu grafikte doğrudan gösteremiyoruz.

PCA ile özellikleri görselleştirme amacıyla iki bileşene indirebiliriz.

PCA'yı ileride daha ayrıntılı işleyeceğiz.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(
    n_components=2
)

X_pca = pca.fit_transform(
    X_scaled
)

pca_df = pd.DataFrame(
    X_pca,
    columns=[
        "PC1",
        "PC2"
    ]
)

pca_df["Kume"] = (
    final_cluster
)

pca_df.head()


# 66. PCA Üzerinde Kümeleri Görmek

In [ ]:
for kume in sorted(
    pca_df["Kume"].unique()
):
    secim = (
        pca_df["Kume"]
        == kume
    )

    plt.scatter(
        pca_df.loc[
            secim,
            "PC1"
        ],
        pca_df.loc[
            secim,
            "PC2"
        ],
        label=f"Küme {kume}"
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA ile Kümelerin 2B Görünümü")
plt.legend()
plt.show()


PCA grafiği görselleştirme için yararlıdır ancak K-Means modelimiz dört standardize özelliğin tamamıyla eğitilmiştir.


# 67. Açıklanan Varyans Oranı

PCA iki bileşenin verideki değişimin ne kadarını temsil ettiğini gösterebilir.


In [ ]:
print(
    pca.explained_variance_ratio_
)

print(
    "Toplam:",
    pca.explained_variance_ratio_.sum()
)


# 68. K-Means'in Güçlü Yönleri

- anlaşılması görece kolaydır,
- hızlı çalışabilir,
- büyük veri kümelerinde kullanılabilir,
- cluster merkezleri üretir,
- yeni örneği kümeye atayabilir,
- segmentasyon problemlerinde yararlıdır.


# 69. K-Means'in Sınırlılıkları

K-Means her veri kümesi için uygun değildir.

Dikkat edilmesi gerekenler:

- K değerini önceden seçmek gerekir,
- uzaklık temellidir,
- ölçeklendirmeden etkilenir,
- aykırı değerlerden etkilenebilir,
- küresel/kompakt kümeleri daha kolay bulur,
- karmaşık şekilli kümelerde zorlanabilir,
- cluster numaraları gerçek sınıf değildir.


# 70. Aykırı Değerler Neden Önemlidir?

K-Means cluster merkezlerini ortalamalar üzerinden oluşturur.

Çok uç değerler merkezlerin konumunu etkileyebilir.

Gerçek projelerde:

- boxplot,
- IQR,
- domain kuralları,
- RobustScaler,
- farklı kümeleme algoritmaları

gibi yöntemler değerlendirilebilir.


# 71. Ölçeklendirmeden K-Means Denemesi

Öğrenme amacıyla scaler kullanmadan da K-Means çalıştıralım.


In [ ]:
olceksiz_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

olceksiz_etiket = (
    olceksiz_model.fit_predict(
        df
    )
)

print(
    "Ölçeksiz silhouette:",
    silhouette_score(
        df,
        olceksiz_etiket
    )
)


Bu skorun ölçeklendirilmiş skorla doğrudan tek başına karşılaştırılması yeterli değildir.

Asıl önemli nokta, uzaklık hesabında büyük ölçekli değişkenlerin etkisinin farklılaşmasıdır.


# 72. Ölçekli Silhouette Skoru

In [ ]:
olcekli_skor = silhouette_score(
    X_scaled,
    final_cluster
)

print(
    "Ölçekli silhouette:",
    olcekli_skor
)


# 73. K-Means Sonucunu CSV'ye Kaydetmek

Cluster etiketlerini veri kümesine ekleyip dışarı aktarabiliriz.


In [ ]:
analiz_df.to_csv(
    "25-kumeleme-sonuclari.csv",
    index=False,
    encoding="utf-8"
)

print(
    "CSV dosyası kaydedildi."
)


# 74. Kaydedilen Dosyayı Tekrar Okumak

In [ ]:
kontrol_df = pd.read_csv(
    "25-kumeleme-sonuclari.csv"
)

kontrol_df.head()


# 75. Scaler ve Modeli Kaydetmek

Yeni veriler için aynı scaler ve aynı K-Means modelini kullanmamız gerekir.

İki nesneyi ayrı ayrı kaydedebiliriz.


In [ ]:
import joblib

joblib.dump(
    scaler,
    "25-kumeleme-scaler.joblib"
)

joblib.dump(
    final_kmeans,
    "25-kmeans-modeli.joblib"
)

print(
    "Scaler ve model kaydedildi."
)


# 76. Pipeline'ı Tek Dosyada Kaydetmek

Daha düzenli yaklaşım olarak scaler + K-Means Pipeline'ını tek dosyada saklayabiliriz.


In [ ]:
joblib.dump(
    kumeleme_pipeline,
    "25-kumeleme-pipeline.joblib"
)

print(
    "Pipeline kaydedildi."
)


# 77. Pipeline'ı Yeniden Yüklemek

In [ ]:
yuklenen_pipeline = joblib.load(
    "25-kumeleme-pipeline.joblib"
)

print(
    yuklenen_pipeline.predict(
        yeni_kayit
    )
)


Güvenilmeyen kaynaklardan gelen pickle/joblib tabanlı model dosyaları yüklenmemelidir.


# 78. Flask ile Kümeleme Uygulaması

Web uygulamasında akış:

```text
HTML Form
↓
Kodlama Süresi
Proje Sayısı
Devam Oranı
Uygulama Puanı
↓
Flask POST
↓
Pipeline.predict()
↓
Küme
↓
Nötr Profil Açıklaması
↓
Jinja
```

şeklinde olabilir.


# 79. Tkinter ile Kümeleme Uygulaması

Masaüstü uygulamasında:

```text
Entry Alanları
↓
Kümeyi Bul Butonu
↓
Pipeline.predict()
↓
Label
```

akışı oluşturulabilir.

Bu nedenle önceki masaüstü derslerimiz yapay zeka uygulamalarının arayüzünde tekrar kullanılabilir.


# 80. API ile Kümeleme

Örnek JSON isteği:

```json
{
    "kodlama_saati": 7,
    "proje_sayisi": 3,
    "devam_orani": 84,
    "uygulama_puani": 79
}
```

cevap:

```json
{
    "kume": 1,
    "profil": "Orta Kodlama Süresi"
}
```

şeklinde olabilir.


# 81. Kümeleme ile Sınıflandırma Arasındaki Fark

### Sınıflandırma

Model eğitim sırasında doğru sınıfları görür.

```text
X + y
```

### Kümeleme

Model doğru sınıfları görmez.

```text
X
```

üzerinden benzerlikleri keşfeder.

Bu nedenle kümeleme etiketleri gerçek sınıf etiketleriyle aynı kavram değildir.


# 82. Cluster Sonucunu Nasıl Yorumlamalıyız?

Doğru yaklaşım:

```text
Bu kümedeki kayıtların ortalama özellikleri şöyledir...
```

Yanlış yaklaşım:

```text
Bu kümedeki kişiler kesin olarak şu tip insandır.
```

Kümeleme verideki matematiksel benzerlikleri gösterir.

İnsanlar hakkında aşırı genelleme üretmek için kullanılmamalıdır.


# 83. K-Means'te Rastgele Başlangıç

K-Means başlangıç cluster merkezlerinden etkilenebilir.

Bu nedenle:

```python
random_state=42
```

ile deneyin tekrar üretilebilir hale getirdik.

Ayrıca:

```python
n_init=10
```

ile algoritmanın farklı başlangıçlardan birden fazla kez çalışmasını sağladık.


# 84. `k-means++`

K-Means varsayılan olarak merkezlerin daha uygun başlangıç noktalarından seçilmesine yardımcı olan `k-means++` yaklaşımını kullanabilir.

İyi başlangıç merkezleri algoritmanın daha iyi çözüme ulaşmasına yardımcı olabilir.


# 85. Kümeleme Sonucunda Başarı Etiketi Yok

Denetimli öğrenmede:

```text
Gerçek y
Tahmin y
```

karşılaştırması yapabiliyorduk.

Kümelemede gerçek sınıf yoksa:

```text
accuracy
precision
recall
```

hesaplamak doğal olarak mümkün değildir.

Bu nedenle:

- inertia,
- silhouette,
- cluster profilleri,
- problem bağlamı

gibi araçlardan yararlanırız.


# 86. Eğer Gerçek Etiketler Varsa?

Bazı araştırma veri kümelerinde gerçek sınıflar ayrıca bulunabilir.

Bu durumda cluster sonucunu gerçek sınıflarla araştırma amacıyla karşılaştıran:

- Adjusted Rand Index,
- Normalized Mutual Information

gibi ölçüler kullanılabilir.

Ancak normal denetimsiz öğrenme senaryosunda gerçek etiket bulunmayabilir.


# 87. Mini Uygulama: Otomatik K Karşılaştırma Fonksiyonu

In [ ]:
def k_karsilastir(
    veri,
    min_k=2,
    max_k=8
):
    sonuclar = []

    for k in range(
        min_k,
        max_k + 1
    ):
        model = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )

        etiket = model.fit_predict(
            veri
        )

        sonuclar.append({
            "K": k,
            "Inertia": model.inertia_,
            "Silhouette": silhouette_score(
                veri,
                etiket
            )
        })

    return pd.DataFrame(
        sonuclar
    )


In [ ]:
otomatik_sonuc = k_karsilastir(
    X_scaled
)

otomatik_sonuc


# 88. Otomatik En İyi Silhouette K Değeri

In [ ]:
en_iyi_k = int(
    otomatik_sonuc.loc[
        otomatik_sonuc[
            "Silhouette"
        ].idxmax(),
        "K"
    ]
)

print(
    "Silhouette'a göre en iyi K:",
    en_iyi_k
)


# 89. Proje Akışı

K-Means projesinde izleyebileceğimiz süreç:

```text
Problem
↓
Etiketsiz Veri
↓
Veri Kalitesi
↓
Özellik Seçimi
↓
Ölçeklendirme
↓
Farklı K Değerleri
↓
Inertia
↓
Elbow
↓
Silhouette
↓
K Seçimi
↓
K-Means
↓
Küme Profilleri
↓
Yeni Örnek
↓
Model / Pipeline Kaydet
```


# 90. Ders Özeti

Bu derste:

- denetimsiz öğrenme,
- kümeleme,
- cluster,
- K-Means,
- K değeri,
- cluster merkezleri,
- `StandardScaler`,
- `fit()`,
- `fit_predict()`,
- `labels_`,
- `cluster_centers_`,
- `predict()`,
- inertia,
- Elbow yöntemi,
- silhouette score,
- uygun K seçimi,
- cluster büyüklükleri,
- cluster profilleri,
- nötr cluster isimlendirme,
- PCA ile görselleştirme,
- Pipeline,
- model kaydetme,
- CSV dışa aktarma,
- Flask / Tkinter / API entegrasyonu

konularını öğrendik.


# 91. Mini Uygulamalar

1. 200 örnekli dört özellikli bir veri kümesi oluşturun.
2. Eksik veri kontrolü yapın.
3. Veriyi `StandardScaler` ile ölçeklendirin.
4. K=2 ile K-Means oluşturun.
5. K=3 ile K-Means oluşturun.
6. K=4 ile K-Means oluşturun.
7. Her modelin inertia değerini bulun.
8. Her modelin silhouette score değerini bulun.
9. K=1 ile K=10 arasında elbow grafiği oluşturun.
10. K=2 ile K=9 arasında silhouette grafiği oluşturun.
11. En yüksek silhouette skorunu otomatik bulun.
12. Cluster etiketlerini DataFrame'e ekleyin.
13. Her kümenin kayıt sayısını bulun.
14. Küme merkezlerini yazdırın.
15. Küme merkezlerini orijinal ölçeğe geri dönüştürün.
16. Küme özellik ortalamalarını hesaplayın.
17. İki özellikle cluster scatter grafiği oluşturun.
18. PCA ile iki boyutlu cluster grafiği oluşturun.
19. Yeni bir örneği cluster'a atayın.
20. Scaler + KMeans Pipeline oluşturun.
21. Pipeline ile yeni örnek tahmini yapın.
22. Sonuçları CSV dosyasına kaydedin.
23. Pipeline'ı `joblib` ile kaydedin.
24. Pipeline'ı yeniden yükleyip tahmin yapın.
25. Cluster sonuçlarını nötr ve açıklayıcı biçimde yorumlayan kısa rapor hazırlayın.


# 92. Yapay Zeka Proje Görevi

Bir **Denetimsiz Öğrenme ve Segmentasyon Projesi** geliştirin.

Konu seçenekleri:

- sensör çalışma örüntüleri,
- ürün özelliklerine göre segmentasyon,
- anonim etkinlik katılım örüntüleri,
- oyun kullanıcı davranışları,
- enerji kullanım profilleri,
- mağaza ürün grupları.

Projede en az:

- 300 kayıt,
- en az 4 sayısal özellik,
- veri kalitesi kontrolü,
- StandardScaler,
- K=2-10 karşılaştırması,
- inertia,
- elbow grafiği,
- silhouette score,
- en uygun K gerekçesi,
- K-Means modeli,
- cluster büyüklükleri,
- cluster merkezleri,
- cluster profil tablosu,
- en az 3 grafik,
- yeni veri için cluster tahmini,
- Pipeline,
- model dosyasına kaydetme,
- kısa analiz raporu

bulunsun.

İnsanlarla ilgili veri kullanılıyorsa cluster'ların gerçek yetenek veya kişilik sınıfları olmadığı açıkça belirtilmelidir.


# Dersin Ana Kazanımı

Bu dersin sonunda öğrencinin aşağıdaki denetimsiz öğrenme zincirini kurabilmesi hedeflenmektedir:

**Etiketsiz Veri**

↓

**Özellik Analizi**

↓

**Ölçeklendirme**

↓

**K-Means**

↓

**Farklı K Değerleri**

↓

**Inertia / Elbow**

↓

**Silhouette Score**

↓

**Küme Seçimi**

↓

**Cluster Profilleri**

↓

**Yeni Veri İçin Küme Tahmini**

↓

**Pipeline ve Model Kaydı**

Bu aşamada öğrenciler yalnızca doğru cevapları verilmiş verilerden öğrenen modeller değil, etiketsiz veriler içindeki yapıyı keşfetmeye çalışan yapay zeka uygulamaları da geliştirebilmektedir.

Bir sonraki derste bu aşamaya kadar öğrendiğimiz NumPy, Pandas, Matplotlib ve scikit-learn bilgilerini kullanarak daha kapsamlı bir **gerçek veriyle yapay zeka tahmin projesi** geliştireceğiz.
